In [1]:
import pandas as pd
import glob
import os
from collections import Counter
from IPython.display import display
import ipaddress
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets


In [2]:
DATA_PATH = r'input'

In [3]:
# ingest event names for specification based detections
with open('events.txt') as f:
    event_names = tuple(line.strip() for line in f if line.strip())

event_names[:5]

('ConsoleLogin', 'CreateAccount', 'ListBuckets')

In [4]:
# ingest raw signal data
def load_file(path):
    df = pd.read_csv(path)
    slash_cols = [c for c in df.columns if c not in ('userAgent', 'userIdentity.arn')]
    bad_slash = df[slash_cols].astype(str).apply(lambda col: col.str.contains('/', na=False)).any(axis=1)
    if bad_slash.any():
        raise ValueError(f"Unexpected '/' in non-userAgent fields in {path}: rows {df.index[bad_slash].tolist()}")
    bad_source = ~df['eventSource'].astype(str).str.endswith('.amazonaws.com')
    if bad_source.any():
        raise ValueError(f"Invalid eventSource values in {path}: {df.loc[bad_source, 'eventSource'].unique().tolist()}")
    if 'rank' in df.columns and pd.to_numeric(df['rank'], errors='coerce').ge(1).any():
        df['type'] = 'ML Anomaly Detection: PYOD'
    elif any(c.startswith('outlier_kmeans') for c in df.columns):
        df['type'] = 'ML Anomaly Detection: K-means'
        df['rank'] = '0'
    return df

files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
signals_raw = pd.concat((load_file(f) for f in files), ignore_index=True)

(
    signals_raw[['type', 'rank', 'readOnly']]
    .value_counts()
    .reset_index()
    .rename(columns={0: 'count'})
    .sort_values(['type', 'rank', 'readOnly'])
    .style.hide(axis='index')
    .set_table_styles([{'selector': 'th', 'props': 'text-align: left; padding: 4px 12px;'},
                       {'selector': 'td', 'props': 'padding: 4px 12px;'}])
    .background_gradient(subset=['count'], cmap='Blues')
)

type,rank,readOnly,count
ML Anomaly Detection: K-means,0,False,86
ML Anomaly Detection: K-means,0,True,45
ML Anomaly Detection: PYOD,1,False,34
ML Anomaly Detection: PYOD,1,True,17
ML Anomaly Detection: PYOD,2,False,14
ML Anomaly Detection: PYOD,2,True,15
ML Anomaly Detection: PYOD,3,False,23
ML Anomaly Detection: PYOD,3,True,13
ML Anomaly Detection: PYOD,4,False,12
ML Anomaly Detection: PYOD,5,False,2


In [5]:
signals_raw.head()

,eventID,eventTime,sourceIPAddress,eventSource,eventName,userAgent,userIdentity.arn,tactic1,tactic2,rank,type,readOnly,outlier_kmeans
0,22741,2018-09-26T18:09:00.000Z,198.18.161.170,billingconsole.amazonaws.com,AWSPaymentInstrumentGateway.Delete,Null,arn:aws:iam::123456789012:root,Defense Evasion,-,5,ML Anomaly Detection: PYOD,False,NaN
1,22742,2018-09-28T13:39:07.000Z,198.18.161.170,signin.amazonaws.com,ConsoleLogin,AWSBFF BFFiOS/1.19.6 (1015) Mobile iOS/11.4.1 ...,arn:aws:iam::123456789012:root,Persistence,Initial Access,3,ML Anomaly Detection: PYOD,False,NaN
2,22743,2018-09-27T15:28:29.000Z,198.18.161.170,signin.amazonaws.com,ConsoleLogin,AWSBFF BFFiOS/1.19.6 (1015) Mobile iOS/11.4.1 ...,arn:aws:iam::123456789012:root,Persistence,Initial Access,4,ML Anomaly Detection: PYOD,False,NaN
3,22744,2018-09-30T17:30:21.000Z,223.0.113.35,signin.amazonaws.com,ConsoleLogin,AWSBFF BFFiOS/1.19.6 (1015) Mobile iOS/11.4.1 ...,arn:aws:iam::123456789012:root,Persistence,Initial Access,3,ML Anomaly Detection: PYOD,False,NaN
4,22745,2018-09-29T13:58:43.000Z,198.18.161.170,signin.amazonaws.com,ConsoleLogin,AWSBFF BFFiOS/1.19.6 (1015) Mobile iOS/11.4.1 ...,arn:aws:iam::123456789012:root,Persistence,Initial Access,3,ML Anomaly Detection: PYOD,False,NaN


In [6]:
# construct detection lattices

# corroborated: events detected by both pyod and kmeans
types_per_event = signals_raw.groupby('eventID')['type'].nunique()
duplicate_ids = types_per_event[types_per_event > 1].index
corroborated = (
    signals_raw[signals_raw['eventID'].isin(duplicate_ids)]
    .drop(columns=['outlier_kmeans_score', 'rank'], errors='ignore')
    .assign(type='Signal Fusion: Model Convergence')
    .drop_duplicates()
    .sort_values('eventID')
    .reset_index(drop=True)
)

# spec_based: events matching specification rules
spec_based = (
    signals_raw[signals_raw['eventName'].isin(event_names)]
    .drop(columns=['rank', 'outlier_kmeans_score'], errors='ignore')
    .assign(type='Specification Based Detection')
    .drop_duplicates()
)

# signal_fusion: spec_based events also caught by ML
ml_ids = signals_raw[signals_raw['type'].isin(['ML Anomaly Detection: PYOD', 'ML Anomaly Detection: K-means:'])]['eventID']
signal_fusion = (
    spec_based[spec_based['eventID'].isin(ml_ids)]
    .drop(columns=['rank', 'outlier_kmeans_score'], errors='ignore')
    .copy()
    .drop_duplicates()
    .assign(**{'type': 'Signal Fusion: ML and Conventional Alert'})
)

# all_detections tuple
all_detections = tuple(
    (row['eventID'], row['type'])
    for df in [corroborated, spec_based, signal_fusion]
    for _, row in df[['eventID', 'type']].drop_duplicates().iterrows()
)

# signals_processed
signals_processed = pd.concat([
    signals_raw.drop(columns=['rank', 'outlier_kmeans_score'], errors='ignore'),
    corroborated,
    spec_based,
    signal_fusion
], ignore_index=True).drop_duplicates()

# per-ARN breakdown with totals
pivot = signals_processed.pivot_table(
    index='userIdentity.arn',
    columns='type',
    aggfunc='size',
    fill_value=0
)
pd.concat([pivot, pivot.sum().rename('Total').to_frame().T])

type,ML Anomaly Detection: K-means,ML Anomaly Detection: PYOD,Signal Fusion: ML and Conventional Alert,Signal Fusion: Model Convergence,Specification Based Detection
arn:aws:iam::123456789012:root,127,127,70,254,70
arn:aws:sts::123456789012:assumed-role/flowlogsRole/vpc-flow-logging+123456789012,4,4,0,8,0
Total,131,131,70,262,70


In [7]:
# search space reduction

EXCLUDE_PRIVATE_IPS = True
EXCLUDE_READONLY = True

def is_private_ip(addr):
    try:
        return ipaddress.ip_address(addr).is_private
    except ValueError:
        return False

sp = signals_processed
if EXCLUDE_PRIVATE_IPS:
    sp = sp[~sp['sourceIPAddress'].apply(is_private_ip)]
if EXCLUDE_READONLY:
    readonly_mask = sp['readOnly'].astype(str).str.lower() == 'true'
    sp = sp[~readonly_mask | sp['eventName'].isin(event_names)]

In [8]:
# arn based filters 
ARN_INCLUDE = ''
ARN_EXCLUDE = ''

if ARN_INCLUDE:
    sp = sp[sp['userIdentity.arn'].astype(str).str.contains(ARN_INCLUDE, case=False, na=False)]
if ARN_EXCLUDE:
    sp = sp[~sp['userIdentity.arn'].astype(str).str.contains(ARN_EXCLUDE, case=False, na=False)]

In [9]:
# compute scores
type_weights = {
    'ML Anomaly Detection: PYOD': 1,
    'ML Anomaly Detection: K-means': 1,
    'Specification Based Detection': 1,
    'Signal Fusion: Model Convergence': 2,
    'Signal Fusion: ML and Conventional Alert': 2,
}

def arn_score(types):
    counts = types.value_counts()
    diversity = sum(type_weights.get(t, 1) * np.log1p(c) for t, c in counts.items())
    volume = np.log1p(len(types))
    return diversity * volume

atomic_weight = (
    sp
    .groupby('userIdentity.arn')['type']
    .apply(arn_score)
    .rename('atomic weight')
    .round(1)
)

atomic_mass = sp.groupby('userIdentity.arn').size().rename('atomic mass')

combined = atomic_weight * atomic_mass
k = combined.max()
scores = (1000 * np.tanh(combined / k)).rename('score').round().astype(int)

threat_cmap = LinearSegmentedColormap.from_list('threat', ['#FFD700', '#FF6600', '#CC0000', '#888888'])

(
    pd.concat([atomic_weight, atomic_mass, scores], axis=1)
    .sort_values('score', ascending=False)
    .style
    .format({'atomic weight': '{:.1f}'})
    .set_table_styles([{'selector': 'th', 'props': 'text-align: left; padding: 4px 12px;'},
                       {'selector': 'td', 'props': 'padding: 4px 12px;'}])
    .background_gradient(subset=['score'], cmap=threat_cmap)
)

,atomic weight,atomic mass,score
userIdentity.arn,,,
arn:aws:sts::123456789012:assumed-role/flowlogsRole/vpc-flow-logging+123456789012,21.6,16,762
arn:aws:iam::123456789012:root,15.1,8,336


In [10]:
# viewer

ALL_FIELDS = [c for c in sp.columns if c != 'userIdentity.arn']
DEFAULT_FIELDS = {'eventName', 'sourceIPAddress', 'type'}

arn_dropdown = widgets.Dropdown(
    options=sorted(sp['userIdentity.arn'].dropna().unique()),
    description='ARN:',
    layout=widgets.Layout(width='600px'),
)

select_all = widgets.Checkbox(value=False, description='Select all', indent=False,
                              layout=widgets.Layout(width='140px'))

field_checkboxes = {
    col: widgets.Checkbox(value=(col in DEFAULT_FIELDS), description=col, indent=False,
                          layout=widgets.Layout(width='220px'))
    for col in ALL_FIELDS
}

html_out = widgets.HTML()
_updating = False

def refresh(_=None):
    fields = [col for col, cb in field_checkboxes.items() if cb.value]
    if not fields:
        html_out.value = '<i>Select at least one field.</i>'
        return
    arn = arn_dropdown.value
    subset = (
        sp[sp['userIdentity.arn'] == arn][fields]
        .value_counts()
        .reset_index()
        .rename(columns={0: 'count'})
        .sort_values(fields)
    )
    html_out.value = (
        subset.style.hide(axis='index')
        .set_table_styles([{'selector': 'th', 'props': 'text-align: left; padding: 4px 12px;'},
                            {'selector': 'td', 'props': 'padding: 4px 12px;'}])
        .background_gradient(subset=['count'], cmap=threat_cmap)
        ._repr_html_()
    )

def on_select_all(change):
    global _updating
    if _updating:
        return
    _updating = True
    for cb in field_checkboxes.values():
        cb.value = change['new']
    _updating = False
    refresh()

def on_field_change(_=None):
    if _updating:
        return
    refresh()

arn_dropdown.observe(refresh, names='value')
select_all.observe(on_select_all, names='value')
for cb in field_checkboxes.values():
    cb.observe(on_field_change, names='value')

refresh()
rows = [widgets.HBox(list(field_checkboxes.values())[i:i+4]) for i in range(0, len(ALL_FIELDS), 4)]
display(arn_dropdown, widgets.HBox([select_all]), *rows, html_out)

Dropdown(description='ARN:', layout=Layout(width='600px'), options=('arn:aws:iam::123456789012:root', 'arn:aws…

HTML(value='<style type="text/css">\n#T_b936e th {\n  text-align: left;\n  padding: 4px 12px;\n}\n#T_b936e td …